# Book 2 — Modeling and Model Evaluation

**"Readers also read…": a Goodreads book recommender** · SIADS 593 Milestone I, Fall 2026

Book 1 produced a 50,000-reader sample at work level, a train/validation/test split by reader, and
the recommendable catalogue (works with at least 10 training likes). This notebook builds the three
recommenders, verifies the fast implementations against the reference implementations, evaluates
all three against baselines on readers the models never saw, and saves the neighbour tables that
**Book 3 (Ranked Fusion Recommender)** combines.

## Model cards

| | Signal | Score for candidate *j* given selected work *i* | Parameters | Build | Output |
|---|---|---|---|---|---|
| **Directional (likes)** | ratings 4–5 | share of *i*'s readers who also liked *j*: `C[i,j] / n[i]` | min shared readers 3; 60 neighbours stored | 20 s | `recommend_directional(work_id, n=5)` |
| **Jaccard (likes)** | ratings 4–5 | overlap of the two audiences: `C[i,j] / (n[i] + n[j] − C[i,j])` | min shared readers 3; 60 neighbours stored | 24 s | `recommend_jaccard(work_id, n=5)` |
| **Finished together (shelves)** | books marked read; no ratings | Jaccard on co-presence among finished books | min shared readers 3; 60 neighbours stored | 43 s | `recommend_finished_together(work_id, n=5)` |
| Baselines | — | most popular; random; most popular in the seed's genre | — | — | same signature |

`C[i,j]` is the number of training readers who have both works under the model's signal; `n[i]` is
the number who have work *i*. The three models are the same computation with a different signal or
a different division, so one engine (`scripts/cooccurrence.py`) builds them all. Every model is
built on the **training readers only**; validation readers tune parameters; test readers are scored
once, at the end. Each model returns the top five `work_id`s for a given work, which is the
interface Book 3 fuses.

**How the notebook runs.** Neighbour tables are saved under `data/processed/neighbours_*.pkl` and
loaded on later runs. The first run takes about four minutes; later runs about two. `SEED = 593`
throughout.

In [1]:
# Setup: Book 1's outputs, the project modules, and the catalogue every model recommends from.
import sys
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

alt.renderers.enable("mimetype")

PROJECT = Path.cwd().resolve().parent if Path.cwd().name == "book2" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT / "scripts"))

import cooccurrence as co          # the engine: reader x work matrices, co-occurrence counts, neighbour tables
import likes_models as lm          # the two likes-based methods: reference implementation, fast wrappers, checks
import finished_together as ft     # the shelf-based model
import evaluation as ev            # one evaluation protocol for every recommender
import model_charts as mc          # the charts in Section 7

PROCESSED = PROJECT / "data" / "processed"
pd.set_option("display.width", 200)
SEED, MIN_LIKES, K, MIN_COUNT, TOP_N = 593, 10, 60, 3, 5

interactions = pd.read_pickle(PROCESSED / "interactions_sample_50k.pkl")
assignment = pd.read_pickle(PROCESSED / "user_splits.pkl")
books = pd.read_pickle(PROCESSED / "book_dimension.pkl")
train_likes = pd.read_pickle(PROCESSED / "train_likes_per_work.pkl")
similar_books = np.load(PROCESSED / "similar_books.npy")

interactions["split"] = interactions["user_id"].map(assignment)
train = interactions[interactions["split"] == "train"]
work_ids = co.recommendable_works(train_likes, floor=MIN_LIKES)
titles = books.set_index("work_id")["title"]
print(f"training readers: {train['user_id'].nunique():,} | recommendable works: {len(work_ids):,}")

training readers: 39,925 | recommendable works: 43,203


## 1 · One engine for three models

The reference implementation computes each score with the same first step: find the readers who
liked the selected work, then count how many of them also liked each other work. Done one work at a
time against the training table, that takes about half a second per question, and evaluating the
models asks tens of thousands of questions. The engine does the same count for every pair of works
at once: a reader × work matrix `A` (1 where the reader has the work under the model's signal), the
product `Aᵀ A` giving the shared-reader count for every pair, and the per-work totals on its
diagonal. Each model is one formula over those counts. Pairs with fewer than 3 shared readers are
dropped (one shared reader is noise), and the 60 best neighbours per work are kept so that filters
applied at query time still leave five results.

In [2]:
# Build the neighbour tables on training readers, or load the saved ones.
def table(name, signal, scorer, min_count=MIN_COUNT):
    path = PROCESSED / f"neighbours_{name}.pkl"
    if path.exists():
        return co.NeighbourTable.load(path)
    built = co.build_neighbour_table(train, work_ids, signal=signal, scorer=scorer, k=K, min_count=min_count)
    built.save(path)
    return built

directional = table("directional_liked", "liked", "directional")
jaccard = table("jaccard_liked", "liked", "jaccard")
finished = table("jaccard_finished", "finished", "jaccard")
print({name: len(t) for name, t in [("directional", directional), ("jaccard", jaccard), ("finished together", finished)]})

{'directional': 43203, 'jaccard': 43203, 'finished together': 43203}


## 2 · The directional score

### Method

In [3]:
# The method, as described in the project's method notebook.
notes = lm.method_discussion()
display(Markdown(notes["step 2: count shared readers"]))
display(Markdown(notes["step 3: directional score"]))

## Step 2: Count Shared Readers for a Selected Work
Both methods start from the same calculation. We identify the readers who liked the selected work, then count how many of those readers also liked each other work. We wrote this calculation once as a helper function so both methods use identical logic.

The function returns the shared reader count for each candidate work along with the number of readers who liked the selected work. It raises an error if no one in the data liked the selected work, since no recommendations can be calculated in that case.

## Step 3: Create the Directional Recommendation Score
For this approach, we identify readers who liked the selected work and find other works those same readers also liked. For each candidate work, we count the number of readers who liked both works and divide that number by the total number of readers who liked the selected work. Multiplying this value by 100 gives us the percentage of readers who liked the selected work who also liked the candidate work.

We use these percentages to rank the recommendations from highest to lowest, excluding the selected work. The score describes shared preferences among readers in our dataset. It does not represent the probability that the person using the dashboard will enjoy the recommended book.

### Reference implementation

The score was first implemented as a pandas function that answers one work at a time. It is kept
unchanged in `scripts/likes_models.py` and serves as the reference the fast version is checked
against.

```python
def directional_recommendations(
    interactions_df,
    selected_item_id,
    item_column="work_id",
    top_n=5,
):
    """Return ranked directional recommendations from positive interactions."""

    recommendations, selected_reader_count = count_shared_readers(
        interactions_df, selected_item_id, item_column
    )

    recommendations["directional_score"] = (
        recommendations["shared_reader_count"] / selected_reader_count * 100
    )

    # Use the item ID to give tied scores a consistent order.
    recommendations = recommendations.sort_values(
        ["directional_score", item_column],
        ascending=[False, True],
    )

    return recommendations.head(top_n).reset_index(drop=True)
```

### Verification

The method notebook included a five-reader, four-work example with the expected scores worked out
by hand: for Work A, candidates B and C share two readers each (50%), D shares one (25%), and ties
are ordered by work id. The engine must reproduce that exactly before it is trusted on real data.

In [4]:
# The engine's tables on the toy data must give the expected orders and scores for both methods.
lm.check_against_toy_example()

,the original order,engine order,the original scores,engine scores
method,,,,
directional,"[B, C, D]","[B, C, D]","[50.0, 50.0, 25.0]","[50.0, 50.0, 25.0]"
jaccard,"[C, B, D]","[C, B, D]","[0.5, 0.4, 0.2]","[0.5, 0.4, 0.2]"


`recommend_directional(work_id, n=5)` reads the precomputed table. To confirm it is the same method,
a table built with no minimum shared-reader count is compared with the reference implementation on
200 randomly chosen seed works from the training data. The reference considers every work a
co-reader liked; the tables hold only the recommendable catalogue, so the reference input is
restricted to the same catalogue for the comparison. Lists must match exactly, including the
tie-break by work id.

In [5]:
# Tables with no minimum shared-reader count, built once for this check (the reference keeps every pair).
directional_all_pairs = table("directional_liked_all_pairs", "liked", "directional", min_count=1)
jaccard_all_pairs = table("jaccard_liked_all_pairs", "liked", "jaccard", min_count=1)
likes = lm.likes_only(train[train["work_id"].isin(work_ids)])   # same candidate set as the tables

# Same 200 seeds, reference versus table: share of identical top-10 lists.
eq = lm.check_equivalence(directional_all_pairs, lm.recommend_directional_reference, likes, work_ids, n_seeds=200, seed=SEED)
lm.summarise_equivalence(eq)

,identical top-10 lists,share identical,mean overlap,first seed that differs
seed works compared,,,,
200,200,1.0,1.0,none


In [6]:
# The output contract: five work_ids for a work. Shown for one well-known seed.
gone_girl = books[books["title"].str.startswith("Gone Girl", na=False)]
seed_work = int(gone_girl.sort_values("ratings_count").iloc[-1]["work_id"])
print(titles[seed_work])
titles[lm.recommend_directional(directional, seed_work, n=TOP_N)].to_frame("directional")

Gone Girl


,directional
work_id,
2792775,"The Hunger Games (The Hunger Games, #1)"
4640799,Harry Potter and the Sorcerer's Stone (Harry P...
3275794,To Kill a Mockingbird
6171458,"Catching Fire (The Hunger Games, #2)"
16827462,The Fault in Our Stars


## 3 · Jaccard similarity

### Method

In [7]:
display(Markdown(notes["step 4: jaccard score"]))

## Step 4: Create the Jaccard Recommendation Function
Jaccard similarity measures the overlap between the groups of readers who liked two works. For each candidate work, we divide the number of readers who liked both works by the number of distinct readers who liked either work.

We calculate the denominator by adding the reader counts for the two works and subtracting the shared-reader count so those readers are not counted twice. Scores range from 0 to 1, with higher scores indicating greater overlap between the two groups of readers.

### Reference implementation

```python
def jaccard_recommendations(
    interactions_df,
    selected_item_id,
    item_column="work_id",
    top_n=5,
):
    """Return ranked Jaccard recommendations from positive interactions."""

    recommendations, selected_reader_count = count_shared_readers(
        interactions_df, selected_item_id, item_column
    )

    # Count all readers of each candidate, including readers outside
    # the selected work's reader group.
    total_reader_counts = (
        interactions_df.groupby(item_column)["user_id"].nunique()
    )

    recommendations["candidate_reader_count"] = (
        recommendations[item_column].map(total_reader_counts)
    )

    recommendations["jaccard_score"] = (
        recommendations["shared_reader_count"]
        / (
            selected_reader_count
            + recommendations["candidate_reader_count"]
            - recommendations["shared_reader_count"]
        )
    )

    recommendations = recommendations.sort_values(
        ["jaccard_score", item_column],
        ascending=[False, True],
    )

    return recommendations.head(top_n).reset_index(drop=True)
```

### Verification

In [8]:
# Equivalence of the fast table with the reference implementation, same 200 seeds.
eq = lm.check_equivalence(jaccard_all_pairs, lm.recommend_jaccard_reference, likes, work_ids, n_seeds=200, seed=SEED)
lm.summarise_equivalence(eq)

,identical top-10 lists,share identical,mean overlap,first seed that differs
seed works compared,,,,
200,200,1.0,1.0,none


In [9]:
titles[lm.recommend_jaccard(jaccard, seed_work, n=TOP_N)].to_frame("jaccard")

,jaccard
work_id,
41107568,The Girl on the Train
6873353,Dark Places
4717423,The Help
3801,Sharp Objects
1708725,"The Girl with the Dragon Tattoo (Millennium, #1)"


The two likes-based scores differ in the denominator only. The directional score divides by the
selected work's readers, so a candidate that everyone has read scores well against every seed.
Jaccard also counts the candidate's own readers, so a candidate has to overlap the seed's audience
*and* not be read by everyone else. Section 7 shows how large that difference is in practice.

## 4 · Finished together

### What was tried

The third model had to use shelf co-presence without ratings, so that the three models differ in
the **signal** they read as well as the formula. Three candidates were built on the training split
and scored on 2,000 validation readers with the protocol in Section 6:

| Candidate | Idea | P@5 | hit@10 | Goodreads hit@10 | Median popularity of recommendations | Distinct works recommended |
|---|---|---|---|---|---|---|
| A. PMI on shelf co-presence | score pairs that share a shelf more often than popularity predicts | 0.033 | 0.22 | 0.42 | 35 readers | 5,331 |
| B. Three-step walk | work → readers → their works → their readers → works, divided by popularity | 0.157 | 0.56 | 0.07 | 10,000 readers | 372 |
| **C. Finished together** | Jaccard on co-presence among books readers marked *read* | **0.237** | **0.74** | 0.38 | 3,800 readers | 3,952 |
| reference: Jaccard on all shelvings | the same formula on every shelved book | 0.212 | 0.74 | 0.45 | 3,600 readers | 3,965 |

These are the numbers from the trial run (2,000 validation readers, a slightly different hold-out
rule). The protocol in Section 6 gives lower absolute values for every model but the same ordering.

The walk collapsed to the same bestsellers for every seed (372 distinct works across 2,000 readers;
*Harry Potter* for *Gone Girl*). PMI went the other way and surfaced obscure co-shelvings that were
mostly noise at three shared readers. Finished-together beat the same formula on all shelvings:
counting only the books readers actually finished removes the to-read pile, which records intention
rather than behaviour.

### Decision

**Finished together: Jaccard similarity on co-presence among works a reader marked as read.** It
uses no rating, it is the strongest of the three candidates on held-out readers, and it gives the
evaluation a clean comparison: the same formula on three signals of increasing commitment, shelved
(reference), finished, liked.

In [10]:
print(ft.describe())
titles[ft.recommend_finished_together(finished, seed_work, n=TOP_N)].to_frame("finished together")

Finished together: Jaccard similarity on co-presence among the works a reader marked as read. Readers who finished both works, divided by readers who finished either. No rating is used. Chosen from three candidates on validation readers: it reached P@5 0.237 against 0.212 for the same formula on all shelvings, 0.033 for PMI, and 0.157 for a three-step walk that recommended only 372 distinct works to 2,000 readers.


,finished together
work_id,
41107568,The Girl on the Train
6873353,Dark Places
4717423,The Help
1708725,"The Girl with the Dragon Tattoo (Millennium, #1)"
3801,Sharp Objects


## 5 · Baselines

Every model's number needs a floor to beat. Three baselines use the same `recommend(work_id, n)`
signature: the most popular works in the training data (the same answer for every seed), random
works from the catalogue, and the most popular works in the seed's genre.

In [11]:
# Baselines built on training readers, and the three models under their display names.
base = ev.baselines(train, work_ids, books, seed=SEED)
models = {
    "Directional (likes)": lambda w, n, exclude: lm.recommend_directional(directional, w, n, exclude),
    "Jaccard (likes)": lambda w, n, exclude: lm.recommend_jaccard(jaccard, w, n, exclude),
    "Finished together (shelves)": lambda w, n, exclude: ft.recommend_finished_together(finished, w, n, exclude),
    **base,
}
list(models)

['Directional (likes)',
 'Jaccard (likes)',
 'Finished together (shelves)',
 'most popular',
 'random',
 'same-genre popular']

## 6 · Evaluation

### Protocol

**Held-out readers.** For each validation or test reader with at least five liked works in the
catalogue, a random 20% of those works are hidden. The remaining 80% are the seeds: each seed's top
neighbours are merged with reciprocal rank fusion (the same rule Book 3 uses) into one list for the
reader, the seeds themselves are removed, and the list is scored against the hidden works:
precision at 5 and 10, recall at 10, hit rate at 10 (did at least one hidden work appear), and NDCG
at 10. No model saw these readers.

**Goodreads' own lists.** For 3,000 catalogue works that have a Goodreads "readers also enjoyed"
list, does the model's top 10 contain any work from that list?

**Catalogue behaviour.** How many distinct works a model ever recommends, and the median popularity
(training readers) of what it recommends compared with the seeds.

### Tuning on validation

In [12]:
# Validation readers: hide 20% of each reader's likes, seed with the rest.
val_readers = assignment.index[assignment == "validation"]
holdout_val = ev.holdout_split_for_readers(interactions, val_readers, work_ids, seed=SEED)
popularity = ev.popularity_from(train)
results_val = ev.evaluate_models(models, holdout_val, similar_books, work_ids, popularity)
SHOW = ["precision@5", "precision@10", "hit@10", "ndcg@10", "Goodreads hit@10", "coverage", "median popularity (recs)"]
results_val[SHOW]

,precision@5,precision@10,hit@10,ndcg@10,Goodreads hit@10,coverage,median popularity (recs)
model,,,,,,,
Directional (likes),0.1368,0.0999,0.5017,0.1457,0.0510,439,10073.0
Jaccard (likes),0.2034,0.1616,0.6779,0.2241,0.3150,6032,3997.0
Finished together (shelves),0.1951,0.1565,0.6774,0.2175,0.3630,6562,4286.0
most popular,0.0786,0.0725,0.4170,0.0960,0.0000,35,10393.0
random,0.0004,0.0003,0.0032,0.0003,0.0013,17765,64.0
same-genre popular,0.0875,0.0637,0.4054,0.0949,0.0000,207,7682.0


Two choices were tested on validation readers: whether to drop pairs with fewer than 3 shared
readers, and whether to remove same-author and same-series works at query time.

In [13]:
# Same-author / same-series filter on, validation readers (the three models only).
related = co.related_works(books, work_ids)
with_filter = {f"{name} + filter": (lambda fn: (lambda w, n, exclude: fn(w, n, exclude | related.get(w, set()))))(fn)
               for name, fn in models.items() if name in mc.MODEL_ORDER}
results_val_filter = ev.evaluate_models(with_filter, holdout_val, similar_books, work_ids, popularity)
results_val_filter[["precision@5", "precision@10", "hit@10", "Goodreads hit@10"]]

,precision@5,precision@10,hit@10,Goodreads hit@10
model,,,,
Directional (likes) + filter,0.1168,0.0911,0.4655,0.0557
Jaccard (likes) + filter,0.1099,0.0941,0.4941,0.4360
Finished together (shelves) + filter,0.1026,0.0891,0.4870,0.5113


### Parameters kept

- **Minimum shared readers = 3.** With no floor, about two thirds of Jaccard links rest on one shared
  reader.
- **Same-author and same-series filter: off for the scores, on for the dashboard.** On validation the
  filter halves precision at 5, because the hidden works are very often the sequel or another book by
  the same author, and it raises agreement with Goodreads' lists, because those lists exclude
  same-author titles. The two targets disagree about sequels. The dashboard applies the filter, since
  a reader of *Gone Girl* knows about *Dark Places*; the reported scores leave it off so the models
  are compared on raw behaviour.
- **60 neighbours stored, 5 returned.** The filter and the removal of a reader's own seeds need
  headroom; 60 was enough for every seed in validation.

### Test readers, scored once

In [14]:
# Test readers: the final numbers.
test_readers = assignment.index[assignment == "test"]
holdout_test = ev.holdout_split_for_readers(interactions, test_readers, work_ids, seed=SEED)
results_test = ev.evaluate_models(models, holdout_test, similar_books, work_ids, popularity)
results_test.to_pickle(PROCESSED / "book2_test_results.pkl")   # full table, all columns
results_test[SHOW]

,precision@5,precision@10,hit@10,ndcg@10,Goodreads hit@10,coverage,median popularity (recs)
model,,,,,,,
Directional (likes),0.1369,0.1008,0.5192,0.1459,0.0510,423,10073.0
Jaccard (likes),0.2080,0.1622,0.6881,0.2279,0.3150,6048,4196.0
Finished together (shelves),0.1991,0.1578,0.6845,0.2215,0.3630,6464,4304.0
most popular,0.0796,0.0737,0.4223,0.0962,0.0000,36,10393.0
random,0.0005,0.0004,0.0039,0.0005,0.0013,17588,64.0
same-genre popular,0.0859,0.0648,0.4137,0.0969,0.0000,210,7682.0


### Results

- **All three models beat every baseline** on held-out readers: precision at 5 of 0.137–0.208 against
  0.080–0.086 for the popularity baselines and 0.0005 for random.
- **The directional score behaves like the popularity baseline.** Its recommendations have a median of
  about 10,000 training readers (most-popular: 10,393; the seeds: 462), it recommends only a few
  hundred distinct works across 4,066 readers, and it rarely matches Goodreads' lists (5%). Dividing by
  the selected work's readers alone favours whatever everyone has read.
- **Jaccard and finished-together are close.** Jaccard on likes is slightly ahead on precision and
  NDCG; finished-together is ahead on agreement with Goodreads' lists and recommends from more of the
  catalogue, without using a rating.
- **Formula matters more than signal here.** Moving from the directional score to Jaccard on the same
  likes changes the results more than moving from likes to finished books with the same formula.

### Limitations

The sample is 50,000 of 876,145 readers, so works with few readers in the full data have fewer
still here, and the recommendable catalogue (43,203 works) is a fraction of what the full data
would support. The held-out test treats a reader's hidden likes as the right answers; a good
recommendation the reader never encountered counts as a miss, so the precision figures understate
quality for every model equally. Goodreads' own lists are the second opinion, but they exclude
same-author titles by construction, which is why the two measures disagree about the filter. The
median popularity of the seeds (462 readers) is reported next to each model's recommendations so
the popularity effect is read against what readers actually liked.

## 7 · Visualization

One panel, two views of the test results. **Left:** the cumulative share of all recommendations each
recommender made to the test readers, against its distinct works ranked by how often they appeared.
The most-popular baseline recommends 36 works in total and the directional score 423, so both curves
reach 100% within a few hundred works; the two Jaccard models spread their recommendations over about
6,000 works each. A curve that rises steeply is a recommender showing most readers the same few books,
whatever they seeded it with. **Right:** the three models' five recommendations for a book chosen from
the dropdown (the 24 most-liked works in the training data), with a bar for how many training readers
have each pick; same-author and same-series titles are removed, as the dashboard does. Across seeds the
directional row repeats the same bestsellers, while the two Jaccard rows change with the seed.

In [15]:
# Left: how many different books each recommender shows (top 10 per test reader, counted across readers).
counts = {name: ev.recommendation_counts(fn, holdout_test) for name, fn in models.items() if name != "random"}

# Right: the three models' five picks for a chosen seed, filter on as on the dashboard.
seed_pool = train_likes.reindex(work_ids).sort_values(ascending=False).head(24).index
rows = []
for seed in seed_pool:
    for name in mc.MODEL_ORDER:
        for rank, w in enumerate(models[name](int(seed), TOP_N, frozenset(related.get(int(seed), set()))), start=1):
            rows.append({"seed": titles[seed][:60], "model": name, "rank": rank, "title": titles[w][:70], "popularity": int(popularity.get(w, 0))})
examples = pd.DataFrame(rows)

mc.two_pane(counts, examples, default_seed=titles[seed_pool[0]][:60])

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## 8 · What Book 3 receives

| File | Contents | Used by |
|---|---|---|
| `data/processed/neighbours_directional_liked.pkl` | `NeighbourTable`: 60 ranked neighbours per recommendable work, directional score on likes | Book 3 fusion |
| `data/processed/neighbours_jaccard_liked.pkl` | same, Jaccard on likes | Book 3 fusion |
| `data/processed/neighbours_jaccard_finished.pkl` | same, Jaccard on finished books | Book 3 fusion |
| `data/processed/book2_test_results.pkl` | the test-reader table above | Book 3 comparison chart |

Each model is `recommend_<name>(table, work_id, n=5, exclude=frozenset())` in its module. Book 3
calls the three, fuses the lists with reciprocal rank fusion, and compares the fused result with
these rows on the same test readers.